# Gemma 4 Developer Agent - offline eval
Runs the official `swegemma.Evaluator` on a deterministic subset of the public tasks for one bundle and writes `results.json`.

In [ ]:
# ---- run configuration (generated by make_notebook.py; edit and re-push) ----
BUNDLE_NAME = 'submission_cand1'
N_TASKS = 20          # size of the deterministic public-task subset
TASK_OFFSET = 0       # start position in the fixed ordering (use for shards / disjoint subsets)
TASK_IDS = []      # explicit instance_ids override N_TASKS/TASK_OFFSET when non-empty
SUBSET_SEED = 'gemma-dev-v1'       # ordering = sorted(sha256(SUBSET_SEED + ':' + instance_id))
CONCURRENCY = 2  # tasks evaluated in parallel against one vLLM server (official --concurrency)
BUNDLE_DIR = ''  # optional: path of an attached dataset holding the bundle; '' = use embedded files
# embedded bundle: relative path -> base64 content (README.md excluded, as in the shipped zip)
EMBEDDED_BUNDLE = {
 "agent.yaml": "bmFtZTogc3dlX3dvcmtmbG93CmFnZW50X2NsYXNzOiBTZXF1ZW50aWFsQWdlbnQKZGVzY3JpcHRpb246IFJlYWQtb25seSBsb2NhbGl6YXRpb24gYW5kIGNvbmNyZXRlIHBsYW4sIGZvbGxvd2VkIGJ5IG9uZSBjb2RlciB0aGF0IGVkaXRzLCB2ZXJpZmllcywgYW5kIHN1Ym1pdHMuCnN1Yl9hZ2VudHM6CiAgLSBjb25maWdfcGF0aDogc3ViX2FnZW50cy9leHBsb3JlLnlhbWwKICAtIGNvbmZpZ19wYXRoOiBzdWJfYWdlbnRzL2NvZGVyLnlhbWwK",
 "configs/explore_sampling.yaml": "dGVtcGVyYXR1cmU6IDAuMQp0b3BfcDogMC45NQptYXhfb3V0cHV0X3Rva2VuczogNDA5Ngp0aGlua2luZ19jb25maWc6CiAgdGhpbmtpbmdfYnVkZ2V0OiAxMDI0CiAgaW5jbHVkZV90aG91Z2h0czogZmFsc2UK",
 "configs/sampling.yaml": "dGVtcGVyYXR1cmU6IDAuMgp0b3BfcDogMC45NQptYXhfb3V0cHV0X3Rva2VuczogODE5Mgp0aGlua2luZ19jb25maWc6CiAgdGhpbmtpbmdfYnVkZ2V0OiAyMDQ4CiAgaW5jbHVkZV90aG91Z2h0czogZmFsc2UK",
 "eval_config.yaml": "ZXZhbHVhdGlvbjoKICB0aW1lb3V0X3NlY29uZHM6IDYwCiAgbWF4X3Rvb2xfY2FsbHM6IDYwCiAgbWF4X3RpbWVfbWludXRlczogNQogIG1heF90dXJuczogMTAwCg==",
 "prompts/explore.md": "TG9jYWxpemUgdGhpcyBpc3N1ZSBpbiBgL3dvcmtzcGFjZWA6IHtwcm9ibGVtX2Rlc2NyaXB0aW9ufQpZb3UgYXJlIHRoZSBSRUFELU9OTFkgZmlyc3Qgc3RhZ2UuIFRoZSBuZXh0IHN0YWdlIHdpbGwgaW1wbGVtZW50IHlvdXIgcGxhbi4KMS4gVXNlIHJ1bl9jb21tYW5kIGZvciBPTkUgYm91bmRlZCByZy9ncmVwIHNlYXJjaCBvZiB0aGUgZXhhY3QgaXNzdWUgc3ltYm9sIGluIGxpa2VseSBzb3VyY2UuIE5vIFB5dGhvbiBzY3JpcHRzLCBubyB0ZXN0cywgbm8gd3JpdGVzLgoyLiBSZWFkIHRoZSBtYXRjaGluZyBzb3VyY2UgcmVnaW9uLiBBIHNlY29uZCBuYXJyb3cgc2VhcmNoL3JlYWQgaXMgYWxsb3dlZCBvbmx5IHRvIHJlc29sdmUgYSBtaXNzaW5nIGRlZmluaXRpb24uCjMuIFNUT1AgdG9vbCB1c2UgYW5kIHJldHVybiBhIGNvbXBhY3QgcGxhbiAoYXQgbW9zdCAxMDAgd29yZHMpOiBzb3VyY2UgcGF0aCBhbmQgc3ltYm9sOyBvYnNlcnZlZCBjYXVzZTsgMeKAkzMgdGlueSBlZGl0czsgb25lIHBhc3MvZmFpbCBhc3NlcnRpb24gYW5kIG9uZSBiZWhhdmlvciB0byBwcmVzZXJ2ZS4gSW5jbHVkZSB0aGUgcmVsZXZhbnQgY3VycmVudCBzb3VyY2UgbGluZXMgc28gdGhlIGNvZGVyIGhhcyBldmlkZW5jZS4gRG8gbm90IHJlcHJvZHVjZSB0aGUgaXNzdWUgb3IgaW1wbGVtZW50IGFueXRoaW5nLiBEbyBub3QgY2FsbCBvdGhlciBhZ2VudHMuCgpHcmFwaCBuYXZpZ2F0aW9uIChyZWFkLW9ubHksIGJlZm9yZSByZXR1cm5pbmcgdGhlIHBsYW4pOgotIFF1ZXJ5IGEgc3BlY2lmaWMgZnVuY3Rpb24gb3IgbWV0aG9kIGZvdW5kIGluIHNvdXJjZSwgcHJlZmVyYWJseSBpdHMgcXVhbGlmaWVkIG5hbWUuIE5ldmVyIHF1ZXJ5IGJyb2FkIGNsYXNzIG5hbWVzIHN1Y2ggYXMgRmFzdEFQSSwgQVBJUm91dGVyLCBvciBCb2R5LiBJZiB0aGUgaXNzdWUgbmFtZXMgb25seSBhIGNsYXNzLCBsb2NhdGUgdGhlIHJlbGV2YW50IG1ldGhvZCBmaXJzdDsgc2tpcCBncmFwaCBsb29rdXAgaWYgbm8gc3BlY2lmaWMgbWV0aG9kIGlzIGtub3duLgotIENhbGwgZ2V0X2NvZGVfbmVpZ2hib3JzIG9uY2Ugd2l0aCB0aGF0IGZ1bmN0aW9uL21ldGhvZCBuYW1lIGFuZCBtYXhfbmVpZ2hib3JzPTguIEl0IHJldHVybnMgbm9kZSBuYW1lcyB3aXRob3V0IGNvZGUuIE9taXQgZWRnZV90eXBlOyBzdG9yZWQgZWRnZXMgdXNlIGxvd2VyY2FzZSBjYWxscy4KLSBZb3UgbWF5IG1ha2UgT05FIGFkZGl0aW9uYWwgZ3JhcGggcXVlcnk6IGdldF9jb2RlX3N1YmdyYXBoKG5vZGVzPTxhdCBtb3N0IDQga25vd24gcXVhbGlmaWVkIGZ1bmN0aW9uL21ldGhvZCBuYW1lcz4pIHRvIGluc3BlY3QgcmVsYXRpb25zaGlwcy4gVGhpcyByZXR1cm5zIG5hbWVzIGFuZCBlZGdlcyB3aXRob3V0IG5vZGUgY29kZS4gSW5zcGVjdCB1c2VmdWwgbmVpZ2hib3JzIHVzaW5nIGEgYm91bmRlZCBzb3VyY2UgcmVhZCByYXRoZXIgdGhhbiByZXRyaWV2aW5nIGZ1bGwgZ3JhcGggbm9kZSBjb2RlLgotIHNlYXJjaF9zaW1pbGFyX2NvZGUgaXMgdW5hdmFpbGFibGUgaGVyZTogayBsaW1pdHMgcmVzdWx0IGNvdW50LCBub3QgdGhlIHNpemUgb2YgYSBub2RlJ3MgY29kZTsgZXZlbiBrPTEgY2FuIGZsb29kIHRoZSBjb250ZXh0LiBHcmFwaHMgYXJlIHN0YXRpYyBoaW50cywgbm90IGNvbXBsZXRlIGNhbGwgZ3JhcGhzLiBFbXB0eSByZXN1bHRzIG1lYW4gY29udGludWUgd2l0aCB0aGUgc291cmNlIGV2aWRlbmNlLCBub3QgcmVwZWF0IHRoZSBxdWVyeS4KLSBDb25maXJtIGFueSBncmFwaC1kZXJpdmVkIGZpeCBjbGFpbSBpbiBhY3R1YWwgc291cmNlLiBLZWVwIHRoZSBzYW1lIGNvbXBhY3QgaGFuZG9mZiBhbmQgc3RvcCBhZnRlciBsb2NhbGl6YXRpb24uIE5vIHdyaXRlcywgcmVwcm9kdWN0aW9uLCBvciB0ZXN0cy4K",
 "prompts/system.md": "SW1wbGVtZW50IHRoaXMgaXNzdWUgaW4gYC93b3Jrc3BhY2VgOiB7cHJvYmxlbV9kZXNjcmlwdGlvbn0KClRoZSByZWFkLW9ubHkgc3RhZ2UgaGFzIGZpbmlzaGVkIGxvY2FsaXphdGlvbi4gSXRzIGV2aWRlbmNlIGFuZCBwbGFuOgp7cmVwYWlyX3BsYW4/fQoKSWYgdGhlIHNhdmVkIHBsYW4gaXMgYWJzZW50LCB1c2UgdGhlIHByZWNlZGluZyBzZWFyY2gvcmVhZCByZXN1bHRzIHRvIGZvcm0gYSBzaG9ydCBjb25jcmV0ZSBwbGFuOyBkbyBub3QgcmVzdGFydCBicm9hZCBleHBsb3JhdGlvbi4KCllvdSBhcmUgdGhlIHNvbGUgQ09ERVIuIENvbXBsZXRlIHRoZSBwcm92aWRlZCBwbGFuIG9uZSBzbWFsbCBzdGVwIGF0IGEgdGltZS4KMS4gUmVhZCB0aGUgbmFtZWQgc291cmNlIHJlZ2lvbiwgdGhlbiBtYWtlIHRoZSBmaXJzdCBtaW5pbWFsIGVkaXRfZmlsZSByZXBsYWNlbWVudC4gRG8gbm90IHJlLXNlYXJjaCB0aGUgcmVwb3NpdG9yeSBvciB3cml0ZSBhIHJlcHJvZHVjdGlvbiBiZWZvcmUgdGhpcyBlZGl0LgoyLiBJbnNwZWN0IHRoZSBtb2RpZmllZCBodW5rIHdpdGggZ2l0IGRpZmYgLS1jaGVjayBhbmQgZ2l0IGRpZmYgLS0gcGF0aC4gSWYgYW5vdGhlciBlZGl0IGlzIG5lZWRlZCwgZG8gaXQgc2VwYXJhdGVseS4KMy4gUnVuIG9uZSBzaG9ydCBpbmxpbmUgUHl0aG9uIGFzc2VydGlvbiAoYXQgbW9zdCA2IGV4ZWN1dGFibGUgbGluZXMpIG9yIG9uZSBleGlzdGluZyB0YXJnZXRlZCB0ZXN0IG5vZGUuIENoZWNrIHRoZSByZXF1ZXN0ZWQgYmVoYXZpb3IgYW5kIG9uZSBwcmVzZXJ2ZWQgYmVoYXZpb3IuIE5vIGNvbW1lbnRzLCBubyB0cnkvZXhjZXB0LCBubyBzY3JhdGNoIGZpbGVzLiBBIGZhaWxlZCBhc3NlcnRpb24gbXVzdCBleGl0IG5vbnplcm8uIEZvciBzZXJpYWxpemF0aW9uIGNvbXBhcmUgdHlwZSBhbmQgZmllbGRzLCBub3QgaWRlbnRpdHkuCjQuIElmIHRoZSBjaGVjayBwYXNzZXMsIHN1Ym1pdF9wYXRjaCBpbW1lZGlhdGVseS4gVGhlbiByZXBvcnQgdGhlIHJlc3VsdCBpbiBvbmUgc2VudGVuY2UuIElmIGl0IGZhaWxzLCBmaXggb25lIHNvdXJjZSBodW5rIGFuZCByZXBlYXQgdGhlIHNhbWUgY2hlY2suCkZvciBhZGRpdGl2ZSBjaGFuZ2VzLCBwcmVzZXJ2ZSBleGlzdGluZyBhbGlhc2VzLCBhY2NlcHRlZCBpbnB1dHMsIGFuZCBkZWZhdWx0cy4gVmVyaWZ5IG9uZSBvbGQgYmVoYXZpb3IgYXMgd2VsbCBhcyB0aGUgbmV3IGJlaGF2aW9yOyBhIG5ldyBmZWF0dXJlIGlzIG5vdCBwZXJtaXNzaW9uIHRvIGRlbGV0ZSBleGlzdGluZyBiZWhhdmlvci4KV29yayBvZmZsaW5lLiBEbyBub3QgaW5zdGFsbCBkZXBlbmRlbmNpZXMsIGNoYW5nZSB0ZXN0cy9jb25maWcsIG9yIHJ1biBmdWxsIHRlc3QgZGlzY292ZXJ5LiBEbyBub3QgcmVwZWF0IHRoZSBwbGFuIGluIHByb3NlLiBUb29sIGNhbGxzIHNob3VsZCBpbXBsZW1lbnQgb25lIHNtYWxsLCB2ZXJpZmlhYmxlIGFjdGlvbi4gVXNlIHRoZSByZW1haW5pbmcgdGFzayBidWRnZXQ7IHJlc2VydmUgdGltZSB0byBzdWJtaXQuCg==",
 "skills/swe-workflow/SKILL.md": "LS0tCm5hbWU6IHN3ZS13b3JrZmxvdwpkZXNjcmlwdGlvbjogQm91bmRlZCBncmVwIHNlYXJjaCwgYSBjb25jcmV0ZSBzbWFsbC1zdGVwIHBsYW4sIG9uZSBlZGl0IGF0IGEgdGltZSwgYW5kIG9ic2VydmFibGUgdmVyaWZpY2F0aW9uLgotLS0KCjEuIFNlYXJjaCB0aGUgaXNzdWUncyBleGFjdCBzeW1ib2xzIHdpdGggc2NvcGVkIHJnL2dyZXAsIHRoZW4gcmVhZCB0aGUgbWF0Y2hpbmcgc291cmNlLgoyLiBTdGF0ZSB1cCB0byB0aHJlZSBjb25jcmV0ZSBzdGVwcyBwbHVzIGEgcGFzcy9mYWlsIGNoZWNrOyBrZWVwIG9uZSBzdGVwIGFjdGl2ZS4KMy4gQXBwbHkgb25lIHNtYWxsIHNvdXJjZSBlZGl0IGFuZCBpbnNwZWN0IGl0cyBkaWZmIGJlZm9yZSB0aGUgbmV4dCBlZGl0Lgo0LiBSdW4gdGhlIGZvY3VzZWQgYXNzZXJ0aW9uL3Rlc3QuIEFuIGV4Y2VwdGlvbiBvciBmYWlsZWQgYXNzZXJ0aW9uIG11c3QgZXhpdCBub256ZXJvLgo1LiBTdWJtaXQgdGhlIHZlcmlmaWVkIHBhdGNoLiBEbyBub3QgZWRpdCB0ZXN0cy9jb25maWcgb3IgcmVwZWF0IGFuIGFscmVhZHkgYW5zd2VyZWQgc2VhcmNoLgo=",
 "skills/verify-before-submit/SKILL.md": "LS0tCm5hbWU6IHZlcmlmeS1iZWZvcmUtc3VibWl0CmRlc2NyaXB0aW9uOiA+CiAgUHJlLXN1Ym1pdCBnYXRlOiB0YXJnZXRlZCB0ZXN0IGV2aWRlbmNlLCBwYXRjaCBoeWdpZW5lLCB0aGVuIHN1Ym1pdF9wYXRjaC4KLS0tCgojIFZlcmlmeSBiZWZvcmUgc3VibWl0CgoxLiBSdW4gKipvbmx5KiogdGhlIHRhcmdldGVkIHRlc3Qgbm9kZSBvciBhIHNob3J0IGFzc2VydGlvbi1iYXNlZCByZXBybyBmb3IgdGhpcyBmaXguIFByaW50ZWQgYm9vbGVhbnMgYW5kIHN3YWxsb3dlZCBleGNlcHRpb25zIGFyZSBub3QgcGFzc2luZyBldmlkZW5jZS4KMi4gYGdpdCBzdGF0dXNgIC8gYGdpdCBkaWZmYDogbm8gc2NyYXRjaCB1bmRlciBgL3dvcmtzcGFjZWAsIG5vIHRlc3QvY29uZmlnIGVkaXRzLgozLiBTY3JhdGNoIGJlbG9uZ3MgaW4gYC90bXBgLgo0LiBPbiBwYXNzaW5nIGFzc2VydGlvbnMgYW5kIGEgY2xlYW4gZGlmZiDihpIgYHN1Ym1pdF9wYXRjaGAg4oaSIHNob3J0IHN1bW1hcnkuCjUuIElmIHRoZSB0YXJnZXRlZCB0ZXN0IGZhaWxzIOKGkiBmaXggc291cmNlIGFnYWluOyBkbyBub3QgZWRpdCB0ZXN0cy4K",
 "sub_agents/coder.yaml": "bmFtZTogc3dlX2FnZW50Cm1vZGVsOiBnZW1tYS00LTMxYi1pdC1xYXQtdzRhMTYtY3QKZGVzY3JpcHRpb246IEFwcGx5IHRoZSBsb2NhbGl6ZWQgcmVwYWlyIGluIHNtYWxsIHN0ZXBzLCB2ZXJpZnksIGFuZCBzdWJtaXQuCmluc3RydWN0aW9uOiAhaW5jbHVkZSAuLi9wcm9tcHRzL3N5c3RlbS5tZApnZW5lcmF0ZV9jb250ZW50X2NvbmZpZzogIWluY2x1ZGUgLi4vY29uZmlncy9zYW1wbGluZy55YW1sCmluY2x1ZGVfY29udGVudHM6IGRlZmF1bHQKdG9vbHM6CiAgLSByZWFkX2ZpbGUKICAtIGVkaXRfZmlsZQogIC0gcnVuX2NvbW1hbmQKICAtIGdldF9zdGF0dXMKICAtIHN1Ym1pdF9wYXRjaApza2lsbHM6CiAgLSBza2lsbHMvc3dlLXdvcmtmbG93CiAgLSBza2lsbHMvdmVyaWZ5LWJlZm9yZS1zdWJtaXQK",
 "sub_agents/explore.yaml": "bmFtZTogZXhwbG9yZQptb2RlbDogZ2VtbWEtNC0zMWItaXQtcWF0LXc0YTE2LWN0CmRlc2NyaXB0aW9uOiBSZWFkLW9ubHkgZ3JlcCBsb2NhbGl6YXRpb24gYW5kIGEgY29uY3JldGUgcmVwYWlyIHBsYW4uCmluc3RydWN0aW9uOiAhaW5jbHVkZSAuLi9wcm9tcHRzL2V4cGxvcmUubWQKZ2VuZXJhdGVfY29udGVudF9jb25maWc6ICFpbmNsdWRlIC4uL2NvbmZpZ3MvZXhwbG9yZV9zYW1wbGluZy55YW1sCm91dHB1dF9rZXk6IHJlcGFpcl9wbGFuCnRvb2xzOgogIC0gcnVuX2NvbW1hbmQKICAtIHJlYWRfZmlsZQogIC0gZ2V0X2NvZGVfbmVpZ2hib3JzCiAgLSBnZXRfY29kZV9zdWJncmFwaAo="
}


In [ ]:
import glob, importlib, os, shutil, subprocess, sys, time, json, hashlib, base64
from pathlib import Path
T0 = time.time()

os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'


def find_dir(candidates, marker_glob):
    for c in candidates:
        if Path(c).is_dir():
            return Path(c)
    hits = sorted(glob.glob(marker_glob, recursive=True))
    if not hits:
        raise FileNotFoundError(f'nothing matches {marker_glob}; candidates {candidates}')
    return Path(hits[0]).parent


WHEELHOUSE_DIR = find_dir(
    ['/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse',
     '/kaggle/input/gemma-4-developer-agent-wheelhouse'],
    '/kaggle/input/**/swegemma-*.whl')
DATA_DIR = find_dir(
    ['/kaggle/input/competitions/gemma-4-developer-agent', '/kaggle/input/gemma-4-developer-agent'],
    '/kaggle/input/**/tasks.jsonl')
MODEL_PATH = find_dir(
    ['/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2',
     '/kaggle/input/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2'],
    '/kaggle/input/**/gemma-4-31b-it-qat-w4a16-ct/**/config.json')
print('wheelhouse', WHEELHOUSE_DIR, '\ndata', DATA_DIR, '\nmodel', MODEL_PATH)

for pth_pattern in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth',
                    '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    target = tmp_whl / target_name
    if not target.exists():
        os.symlink(w, target)
wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
importlib.invalidate_caches()
T_INSTALL = time.time() - T0
print(f'installed in {T_INSTALL:.0f}s')


In [ ]:
import yaml
from swegemma.models import load_tasks

WORKING_DIR = Path('/kaggle/working')
if BUNDLE_DIR:
    hits = sorted(glob.glob(f'{BUNDLE_DIR}/**/agent.yaml', recursive=True), key=lambda s: s.count('/'))
    AGENT_DIR = Path(hits[0]).parent
else:
    AGENT_DIR = WORKING_DIR / 'bundle'
    if AGENT_DIR.exists():
        shutil.rmtree(AGENT_DIR)
    for rel, b64 in EMBEDDED_BUNDLE.items():
        p = AGENT_DIR / rel
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_bytes(base64.b64decode(b64))
bundle_sha = hashlib.sha256(b''.join(
    rel.encode() + (AGENT_DIR / rel).read_bytes()
    for rel in sorted(str(p.relative_to(AGENT_DIR)) for p in AGENT_DIR.rglob('*') if p.is_file()))).hexdigest()
print('bundle', BUNDLE_NAME, AGENT_DIR, 'sha', bundle_sha[:12])
for p in sorted(AGENT_DIR.rglob('*')):
    if p.is_file():
        print('  ', p.relative_to(AGENT_DIR), p.stat().st_size)

# official validation, as on the scorer
from swegemma.config import build_submission_limits
try:
    from adk_submission import validate_directory
    validate_directory(AGENT_DIR, build_submission_limits()[0])
    print('validate_directory: OK')
except Exception as e:  # report, do not abort: the Evaluator re-validates
    print('validate_directory:', type(e).__name__, e)

TASKS_PATH = DATA_DIR / 'tasks.jsonl'
all_tasks = load_tasks(TASKS_PATH)
order = sorted(all_tasks, key=lambda t: hashlib.sha256(f'{SUBSET_SEED}:{t.instance_id}'.encode()).hexdigest())
if TASK_IDS:
    SELECTED = [t.instance_id for t in all_tasks if t.instance_id in set(TASK_IDS)]
else:
    SELECTED = [t.instance_id for t in order[TASK_OFFSET:TASK_OFFSET + N_TASKS]]
print(f'{len(all_tasks)} public tasks; selected {len(SELECTED)}:', SELECTED)

raw_eval_cfg = yaml.safe_load((AGENT_DIR / 'eval_config.yaml').read_text()) if (AGENT_DIR / 'eval_config.yaml').exists() else {}
eval_section = (raw_eval_cfg or {}).get('evaluation', raw_eval_cfg or {})
timeout_seconds = int(eval_section.get('timeout_seconds', 300))
max_tool_calls = int(eval_section.get('max_tool_calls', 100))
max_time_minutes = float(eval_section.get('max_time_minutes', 60.0))
turns_raw = eval_section.get('max_turns', eval_section.get('max_llm_calls'))
max_turns = int(turns_raw) if turns_raw is not None else None
CAPS = dict(timeout_seconds=timeout_seconds, max_tool_calls=max_tool_calls,
            max_time_minutes=max_time_minutes, max_turns=max_turns)
print('caps from eval_config:', CAPS)


In [ ]:
import litellm, torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
from swegemma.models.discovery import validate_single_declared_model

litellm.drop_params = True
TARGET_MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'
declared_model = validate_single_declared_model(AGENT_DIR)
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
print('gpus', gpu_count, [torch.cuda.get_device_name(i) for i in range(gpu_count)])

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH), port=8000, host='127.0.0.1',
    tool_call_parser='gemma4', reasoning_parser='gemma4', max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90, enable_auto_tool_choice=True, enable_lora=True,
    max_loras=8, max_lora_rank=128, tensor_parallel_size=tp_size, startup_timeout=60 * 20,
)
t = time.time()
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
T_VLLM = time.time() - t
print(f'vLLM up on {server_instance.base_url} (tp={tp_size}) in {T_VLLM:.0f}s')
models = server_instance.create_model_registry(
    aliases=[declared_model, TARGET_MODEL_NAME], model_prefix='openai/', api_key='EMPTY')


In [ ]:
import asyncio, concurrent.futures, logging
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator
logging.basicConfig(level=logging.WARNING)


def run_sync(fn):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())


RESULTS_DIR = WORKING_DIR / 'results'
limits, gen_constraints = build_submission_limits()
eval_config = EvalConfig(
    tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / 'snapshots', results_dir=RESULTS_DIR,
    submission_dir=AGENT_DIR, models=models, sandbox='subprocess',
    timeout_seconds=timeout_seconds, max_time_minutes=max_time_minutes,
    max_tool_calls=max_tool_calls, max_turns=max_turns,
    limits=limits, generation_constraints=gen_constraints, adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=15, overlap_size=2, token_threshold=14336, event_retention_size=5),
    graph_dir=str(DATA_DIR / 'graphs'), embeddings_dir=str(DATA_DIR / 'embeddings'),
    wheels_dir=DATA_DIR / 'wheels', verbose=False,
    task_ids=SELECTED, concurrency=CONCURRENCY, display_mode='quiet',
)
evaluator = Evaluator(eval_config)
t = time.time()
eval_res = run_sync(evaluator.run)
T_EVAL = time.time() - t
print(f'evaluated {len(eval_res.task_results)} tasks in {T_EVAL:.0f}s')


In [ ]:
def _tail(s, n=1500):
    s = s or ''
    return s[-n:]

per_task = []
for r in eval_res.task_results:
    d = r.model_dump(exclude={'trace', 'agent_patch', 'test_output'})
    row = {
        'instance_id': r.instance_id,
        'repo': r.repo,
        'resolved': bool(r.resolved),
        'duration_seconds': round(float(getattr(r, 'duration_seconds', 0) or 0), 1),
        'tool_calls': getattr(r, 'tool_calls', None),
        'llm_calls': getattr(r, 'total_llm_calls', None),
        'test_exit_code': getattr(r, 'test_exit_code', None),
        'patch_chars': len(r.agent_patch or ''),
        'error': getattr(r, 'error_message', None) or d.get('error'),
        'test_output_tail': _tail(r.test_output, 800),
    }
    for k, v in d.items():  # keep any other scalar fields the harness reports
        if k not in row and isinstance(v, (int, float, str, bool, type(None))):
            row[k] = v
    per_task.append(row)

n = len(per_task)
n_pass = sum(r['resolved'] for r in per_task)
t_total = time.time() - T0
summary = {
    'bundle': BUNDLE_NAME, 'bundle_sha256': bundle_sha, 'caps': CAPS,
    'subset_seed': SUBSET_SEED, 'task_offset': TASK_OFFSET, 'concurrency': CONCURRENCY,
    'n_tasks': n, 'n_resolved': n_pass, 'pass_rate': (n_pass / n) if n else 0.0,
    'n_empty_patch': sum(r['patch_chars'] == 0 for r in per_task),
    'n_errors': sum(bool(r['error']) for r in per_task),
    'mean_task_seconds': (sum(r['duration_seconds'] for r in per_task) / n) if n else None,
    'seconds': {'install': round(T_INSTALL), 'vllm_start': round(T_VLLM), 'eval': round(T_EVAL), 'total': round(t_total)},
    'machine_minutes_per_task_eval_only': round(T_EVAL / 60 / max(n, 1), 2),
    'machine_minutes_per_task_incl_startup': round(t_total / 60 / max(n, 1), 2),
}
out = {'summary': summary, 'tasks': per_task}
(WORKING_DIR / 'results.json').write_text(json.dumps(out, indent=2, default=str))
print(json.dumps(summary, indent=2))
for r in per_task:
    print(f"{'PASS' if r['resolved'] else 'fail'} {r['instance_id']:<28} {r['duration_seconds']:>6.0f}s "
          f"tools={r['tool_calls']} llm={r['llm_calls']} patch={r['patch_chars']} err={(r['error'] or '')[:80]}")


In [ ]:
try:
    server_instance.stop()
except Exception as e:
    print('stop:', e)
# keep outputs small: drop trace dumps bigger than 2 MB
for p in (WORKING_DIR / 'results').rglob('*'):
    if p.is_file() and p.stat().st_size > 2_000_000:
        p.unlink()
if (WORKING_DIR / 'bundle').exists():
    shutil.rmtree(WORKING_DIR / 'bundle')
